# Capstone --- Chapter 10: Trajectory Evaluation and Metrics

Chapter~10 argues that a single scalar cannot describe an agent run. A run is scored at three levels: the *decision* it reached, the *structure* of the trajectory that produced it, and the *process health* of that trajectory; and a claim's *groundedness* is measured as a distance from its supporting evidence. This companion reads those levels on the capstone banking complaint agent, using the evaluation functions in `agentlab.evaluation` and the pinned campaign artifact `data/capstone_run.json`.

The metric layer is a set of pure functions over a `Trajectory`. Each is independent and composable, so a report is assembled by applying them rather than by threading a single accumulator through the run. Reading the module's exports names the vocabulary the chapter defines.

In [ ]:
import forgeloop.agents.evaluation as ev

level_metrics = ['task_success', 'escalated']            # decision level
structure_metrics = ['step_count', 'tool_call_count']    # trajectory structure
health_metrics = ['tool_failure_count', 'finished_cleanly']  # process health
for group, names in [('decision', level_metrics),
                     ('structure', structure_metrics),
                     ('process health', health_metrics)]:
    present = [n for n in names if hasattr(ev, n)]
    print(f'{group:16s}: {present}')

## The pinned campaign artifact

A single run yields one trajectory; a *campaign* aggregates many runs into the metrics the chapter reports. The capstone's campaign is pinned to `data/capstone_run.json` so its numbers are stable across readings. The top-level keys separate the decision-level score from the structural and per-component diagnostics.

In [ ]:
import json
from pathlib import Path

root = next((c for c in (Path('.'), Path('..'), Path('../code'), Path('code')) if (c / 'data').exists()), Path('.'))
run = json.loads((root / 'data' / 'capstone_run.json').read_text())
print('n_runs           :', run['n_runs'])
print('top-level keys   :', list(run.keys()))

## The decision level

The decision-level score is the fraction of test queries decisioned correctly: the right classification together with the right escalate / do-not-escalate decision. It is deliberately narrow. The artifact records its definition alongside the number so the two are never separated.

In [ ]:
overall = run['overall']
print('decision accuracy:', overall['accuracy'])
print('definition       :', overall['definition'])

## The trajectory-structure level

The chapter distinguishes reaching the right decision from following the intended path to it. `workflow_adherence` measures whether the trajectory visited the tools in the mandated order, and `audit_verifies` records whether the run's audit trail reconstructs. A run may decide correctly on a malformed trajectory, so these are reported apart from accuracy rather than folded into it.

In [ ]:
print('workflow_adherence:', run['workflow_adherence'])
print('audit_verifies    :', run['audit_verifies'])
print('weak_link         :', run['weak_link'])

## Per-component attribution

A decision accuracy of $0.675$ is a property of the whole workflow. Attribution asks which tool the errors concentrate in. Each per-tool entry reports how many outputs were scored, how many were correct, and the resulting accuracy, so the aggregate is decomposed into the components that produced it.

In [ ]:
for name, stats in run['per_tool'].items():
    acc = stats['accuracy']
    print(f'{name:20s} accuracy={acc:.3f}  '
          f'({stats["correct"]}/{stats["scored"]} scored)')

The `weak_link` count locates the errors by component, so a low aggregate is traced to the tool that produced most of the failed decisions rather than attributed to the agent as a whole.

In [ ]:
weak = run['weak_link']
for tool, failures in sorted(weak.items(), key=lambda kv: -kv[1]):
    print(f'{tool:20s} failures={failures}')

## Groundedness as a distance

Whether a drafted claim is *supported by* the retrieved evidence is a distinct question from whether the decision was correct. Chapter~10 treats groundedness as a distance between a claim and its nearest evidence: a claim close enough to some evidence is `SUPPORTED`, and one too far from all of it is `UNSUPPORTED`. `check_groundedness` returns that verdict together with the evidence it matched.

In [ ]:
from forgeloop.agents.evaluation import Claim, groundedness_report

evidence = {
    'reg-e': ('Regulation E limits consumer liability for unauthorized electronic '
              'fund transfers and requires timely error resolution.'),
    'overdraft': ('Overdraft fees may be refunded when the charge was not '
                  'authorized by the accountholder.'),
}
grounded = Claim(text='Regulation E limits liability for unauthorized electronic fund transfers.')
fabricated = Claim(text='The bank guarantees a refund within twenty four hours for any dispute.')
for r in groundedness_report([grounded, fabricated], evidence):
    print(f'{r.verdict.value:12s} evidence={str(r.evidence_id):10s} | {r.claim}')

The supported claim resolves to the evidence it paraphrases; the fabricated guarantee matches nothing and is `UNSUPPORTED` with no evidence identifier. This is the distance made operational: the verdict is a function of how near the claim sits to the closest supporting passage, not of how fluent the draft reads.

## Reading the levels together

This is the capstone's realization of Chapter~10. The three levels answer different questions and do not substitute for one another: decision accuracy is $0.675$, workflow adherence is $1.0$, and the errors concentrate in `classify_complaint`, while groundedness is scored per claim as a distance from evidence. Chapter~16 assembles the five tools into the governed workflow these metrics score, and Chapter~17 turns the campaign into a designed test suite that attributes the aggregate to the factors that move it.